In [1]:
import numpy as np

In [2]:
class SVM:
  def __init__(self, X, y, alpha, kernel='linear', b=0, max_iter=300, C=1):
    self.X = X
    self.y = y
    self.alpha = alpha
    self.b = b
    self.C = C
    self.kernel = kernel
    self.max_iter = max_iter

  def linear_kerenel(self, xi, xj):
    return np.inner(xi, xj)

  def gaussian_kernel(self, x1, x2, sigma=1):
    if np.ndim(x1) == 1 and np.ndim(x2) == 1:
      return np.exp(-(np.linalg.norm(x1-x2,2))**2/(2*sigma**2))
    elif(np.ndim(x1)>1 and np.ndim(x2) == 1) or (np.ndim(x1) == 1 and np.ndim(x2)>1):
      return np.exp(-(np.linalg.norm(x1-x2, 2, axis=1)**2)/(2*sigma**2))
    elif np.ndim(x1) > 1 and np.ndim(x2) > 1 :
        return np.exp(-(np.linalg.norm(x1[:, np.newaxis] \
                             - x2[np.newaxis, :], 2, axis = 2) ** 2)/(2*sigma**2))
    return 0

  def decide_kernel(self, xi, xj):
    if self.kernel == 'linear':
        return self.linear_kerenel(xi, xj)
    elif self.kernel == 'gaussian':
        return self.gaussian_kernel(xi, xj)

  def make_prediction(self, kernel, alpha, x, x_i, y, b):
    return np.dot(alpha*y, kernel(x_i, x)) + b

  def compute_L(self, alphai, alphaj, C, y_i, y_j):
    if y_i != y_j:
        return max(0, alphaj - alphai)
    else:
        return max(0, alphai + alphaj - C)

  def compute_H(self, alphai, alphaj, C, y_i, y_j):
    if y_i != y_j:
        return min(C, C + alphaj - alphai)
    else:
        return min(C, alphai + alphaj)

  def calc_k(self,x_i, x_j, kernel):
    return (kernel(x_i,x_i)+kernel(x_j,x_j) - 2*kernel(x_i,x_j))

  def calc_error(self,fx_i, y_i):
      return fx_i - y_i

  def update_alpha_j(self,y_j, fx_j, y_i, fx_i ,k, L , H):
    return (y_j*(self.calc_error(fx_i, y_i) - self.calc_error(fx_j, y_j))/k)

  def clip_alpha_j(self,alpha_j, H, L):
      if alpha_j > H:
          return H
      elif alpha_j < L:
          return L
      else:
          return alpha_j

  def update_alpha_i(self, alpha_j_new, alpha_j_old, y_i, y_j):
    return (y_i * y_j) * (alpha_j_old - alpha_j_new)

  def update_b(self,error_1, error_2, x_i, x_j, y_i,y_j, kernel, alpha_old, alpha, b):
    b_one = -error_1 - (y_i * (-alpha_old[0] + alpha[0]))*kernel(x_i, x_i) - (y_j * (-alpha_old[1] + alpha[1]))*kernel(x_i, x_j) + b
    b_two = -error_2 - (y_i * (-alpha_old[0] + alpha[0]))*kernel(x_i, x_i) - (y_j * (-alpha_old[1] + alpha[1]))*kernel(x_i, x_j) + b
    return b_one, b_two

  def decide_b(self,C, alpha_new, b_one, b_two):
    if 0<alpha_new[0]<C:
      return b_one
    elif 0<alpha_new[1]<C:
      return b_two
    else:
      return (b_one + b_two)/2
  def fit(self):
    iterations = 0
    while iterations < self.max_iter:
      num_changed = 0

      for i in range(len(self.X)):
        x_i = self.X[i]
        y_i = self.y[i]
        alpha_i = self.alpha[i]
        fx_i = self.make_prediction(self.decide_kernel, self.alpha,self.X,x_i, self.y, self.b)
        E_i = self.calc_error(fx_i, y_i)

        j = np.argmax(np.abs(E_i - np.array([self.calc_error(self.make_prediction(self.decide_kernel, self.alpha, self.X, self.X[k], self.y, self.b), self.y[k]) for k in range(len(self.X))])))
        x_j = self.X[j]
        y_j = self.y[j]
        alpha_j = self.alpha[j]
        fx_j = self.make_prediction(self.decide_kernel, self.alpha,self.X,x_j, self.y, self.b)
        E_j = self.calc_error(fx_j, y_j)

        L = self.compute_L(alpha_i,alpha_j,self.C,y_i,y_j)
        H = self.compute_H(alpha_i,alpha_j,self.C,y_i,y_j)
        k = self.calc_k(x_i, x_j, self.decide_kernel)
        if L==H or k<=0:
          continue
        alpha_i_old, alpha_j_old = alpha_i, alpha_j
        alpha_j_new = alpha_j + self.update_alpha_j(y_j, fx_j, y_i, fx_i ,k, L , H)
        alpha_j_new = self.clip_alpha_j(alpha_j_new, H, L)
        alpha_i_new = alpha_i+self.update_alpha_i(alpha_j_new,alpha_j_old,y_i,y_j)
        b1,b2 = self.update_b(E_i, E_j, x_i, x_j, y_i,y_j, self.decide_kernel, [alpha_i_old, alpha_j_old], [alpha_i_new, alpha_j_new], self.b)
        self.b = self.decide_b(self.C,[alpha_j_new, alpha_i_new],b1,b2)
        self.alpha[i] = alpha_i_new
        self.alpha[j] = alpha_j_new
        num_changed+=1
      if num_changed == 0:
            break
      iterations+=1
    self.w = (self.y * self.alpha)@(self.X)
  def predict(self, X_train):
    return self.w.T@X_train + self.b


In [3]:
X = np.array([
    [2, 3],
    [3, 3],
    [2, 4],
    [3, 4],
    [4, 5],
    [1, 1],
    [2, 1],
    [1, 2],
    [2, 2],
    [3, 1]])

y = np.array([1, 1, 1, 1, 1, -1, -1, -1, -1, -1])
alpha = np.zeros(len(X))
print(X.shape)
print(y.shape)
print(alpha)
print(X)

(10, 2)
(10,)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[[2 3]
 [3 3]
 [2 4]
 [3 4]
 [4 5]
 [1 1]
 [2 1]
 [1 2]
 [2 2]
 [3 1]]


In [4]:
model = SVM(X,y,alpha)

In [5]:
model.fit()

In [6]:
print(model.alpha)
print(model.w)
print(model.b)

[0.4 0.  0.  0.  0.  0.4 0.  0.  0.  0. ]
[0.4 0.8]
-3.8000000000000007


In [7]:
if model.predict([0.1,0.1])>0:
  output=1
else:
  output = -1
print(output)

-1
